In [1]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [6]:
#from utils import dehomogenize, homogenize, draw_epipolar, visualize_pcd
import numpy as np
import scipy
import cv2 
import os
import matplotlib.pyplot as plt

In [7]:
#utils module
# images are tiny otherwise
plt.rcParams['figure.figsize'] = [15, 15]


def homogenize(X):
    """
    Given real NxK points, return Nx(K+1) homogeneous points
    """
    return np.hstack([X, np.ones((X.shape[0], 1))])


def dehomogenize(X):
    """
    Given homogeneous NxK points, return Nx(K-1) real points
    """
    K = X.shape[0]
    return X[:, :(K - 1)] / X[:, (K - 1)][:, None]


def ncolors(n):
    """
    Generate a consistent set of n colors that's somewhat evenly spaced.
    Uses the classic sample uniformly on H trick plus different saturations
    to get some more colors.

    Return a list of lists of ints [(r,g,b), ...]
    """
    def ngaps(n):
        # Generate: 1/2, 1/4, 3/4, 1/8, 3/8, ...
        vals = []
        num, den = 1, 2
        while len(vals) < n:
            vals.append((1.0 * num) / den)
            num += 2
            if num > den:
                num, den = 1, den * 2
        return vals

    colors = []

    # number of saturations to use
    numSats = 3 if n > 10 else 1

    # number of hues (we'll cycle through this numSats times)
    numHues = n // numSats
    numHues += 1 if ((numHues * numSats) < n) else 0

    gaps = ngaps(numHues)
    for i in range(n):
        h = gaps[i % numHues] * 180  # where in the hue wheel we are
        satInd = i // numHues  # which saturation index we're onto
        sat = (satInd + 1.0) / (numSats) * 255  # saturation
        # convert hsv -> rgb
        rgb = cv2.cvtColor(np.uint8([[[h, sat, 255]]]), cv2.COLOR_HSV2RGB)
        # force it to be a list of ints; opencv's picky
        colors.append(list([int(v) for v in rgb.flatten()]))
    return colors


def drawlines(img1, img2, lines, pts1, pts2):
    """
    Draw points and epipolar lines

    Substantially Improved version of opencv documentation
    https://docs.opencv.org/master/da/de9/tutorial_py_epipolar_geometry.html

    This:
    - uses a consistent set of colors, which should make debugging easier
    - doesn't blow up with vertical lines
    """

    r, c = img1.shape
    img1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)
    img2 = cv2.cvtColor(img2, cv2.COLOR_GRAY2BGR)
    colorsUse = ncolors(len(lines))

    H, W = img1.shape[0], img1.shape[1]

    X, Y = np.meshgrid(np.arange(W).astype(np.float),
                       np.arange(H).astype(np.float))

    for ii, (r, pt1, pt2) in enumerate(zip(lines, pts1, pts2)):
        color = colorsUse[ii]

        # r is [a,b,c]; draw the line ax+by+c = 0 by finding its intersection
        # with the bounding rectangle.  If it's in the image, r will cross it
        toCheck = [np.cross([0, 0, 1], [W, 0, 1]),
                   np.cross([0, 0, 1], [0, H, 1]),
                   np.cross([W, H, 1], [W, 0, 1]),
                   np.cross([W, H, 1], [0, H, 1])]

        # find the intersection with all the boundaries
        isects = []
        for i in range(len(toCheck)):
            isect = np.cross(toCheck[i], r)
            isects.append(isect[:2] / isect[2])
        # sort by distance to center of image
        isects.sort(key=lambda v: np.linalg.norm(v - np.array([W / 2, H / 2])))
        x0, y0 = map(int, isects[0])
        x1, y1 = map(int, isects[1])

        img1 = cv2.line(img1, (x0, y0), (x1, y1), color, 5)
        img1 = cv2.circle(img1, tuple(map(int, pt1)), 10, color, -1)
        img2 = cv2.circle(img2, tuple(map(int, pt2)), 10, color, -1)

    return img1, img2


def draw_epipole(img, epiH):
    # do not do anything if it is undefined or is exactly 0. Everything else
    # goes through ok -- even if the dehomogenizing makes it infinite, that's
    # fine and we'll catch it later
    if epiH is None or epiH[2] == 0:
        return img

    epi = epiH[:2] / epiH[2]
    H, W = img.shape[0], img.shape[1]
    # check to see if we can even draw it

    if (0 <= epi[0] < W) and (0 <= epi[1] < H):
        epi = tuple(map(int, epi))
        img = cv2.circle(img, epi, 20, (0, 0, 0), -1)
        img = cv2.circle(img, epi, 10, (255, 255, 255), -1)
    return img


def draw_epipolar(img1, img2, F, pts1, pts2,
                  epi1, epi2, filename):
    """
    Improved version of opencv documentation
    https://docs.opencv.org/master/da/de9/tutorial_py_epipolar_geometry.html

    Inputs:
    - img1: image1
    - img2: image2
    - F: the fundamental matrix
    - epi1: the epipole for image 1 in homogeneous coordinates
    - epi2: the epipole for image 2 in homogeneous coordinates
    - filename: if is None, then plt.show() else save it

    """

    lines1 = cv2.computeCorrespondEpilines(pts2.reshape(-1, 1, 2), 2, F)
    lines1 = lines1.reshape(-1, 3)
    img1_with_lines, img2_points = drawlines(img1, img2, lines1, pts1, pts2)
    img1_with_lines = draw_epipole(img1_with_lines, epi1)

    lines2 = cv2.computeCorrespondEpilines(pts1.reshape(-1, 1, 2), 1, F)
    lines2 = lines2.reshape(-1, 3)
    img2_with_lines, img1_points = drawlines(img2, img1, lines2, pts2, pts1)
    img2_with_lines = draw_epipole(img2_with_lines, epi2)

    plt.figure()
    plt.subplot(221)
    plt.imshow(img1_points)
    plt.title("Image 1 points")
    plt.subplot(222)
    plt.imshow(img2_with_lines)
    plt.title("Image 2 points and epipolar lines from image 1")

    plt.subplot(223)
    plt.imshow(img1_with_lines)
    plt.title("Image 1 points and epipolar lines from image 2")

    plt.subplot(224)
    plt.imshow(img2_points)
    plt.title("Image 2 points")

    plt.tight_layout()

    if filename is None:
        plt.show()
    else:
        plt.savefig(filename)
    plt.close()


def visualize_pcd(points, filename=None):
    """
    Visualize the point cloud.

    Inputs:
    - points: the matrix of points
    - filename: if None, plt.show() else save it here

    """
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(points[0].T, points[2].T, points[1].T)
    plt.tight_layout()

    if filename is None:
        plt.show()
    else:
        plt.savefig(filename)
    plt.close()


In [3]:
def Normalize(pts1):
    mean_shift = np.zeros(pts1.shape)
    mean0 = np.mean(pts1[:,0])
    mean1 = np.mean(pts1[:,1])
    mean_shift[:,0] = pts1[:,0] - mean0
    mean_shift[:,1] = pts1[:,1] - mean1
    N = len(pts1)
    s1 = (N*2)**0.5/(sum([((x-mean1)**2 + (y-mean1)**2)**0.5 for x,y in mean_shift]))
    T1 = np.array([[s1, 0, -mean0*s1], [0, s1, -mean1*s1], [0, 0, 1]])
    #T2 = np.array([[s2, 0, -mean2[0]*s2], [0, s2, -mean2[1]*s2], [0, 0, 1]])
    return T1
       
    
def find_fundamental_matrix(shape, pts1, pts2):
    """
    Computes Fundamental Matrix F that relates points in two images by the:

        [u' v' 1] F [u v 1]^T = 0
        or
        l = F [u v 1]^T  -- the epipolar line for point [u v] in image 2
        [u' v' 1] F = l'   -- the epipolar line for point [u' v'] in image 1

    Where (u,v) and (u',v') are the 2D image coordinates of the left and
    the right images respectively.

    Inputs:
    - shape: Tuple containing shape of img1
    - pts1: Numpy array of shape (N,2) giving image coordinates in img1
    - pts2: Numpy array of shape (N,2) giving image coordinates in img2

    Returns:
    - F: Numpy array of shape (3,3) giving the fundamental matrix F
    """
    ###########################################################################
    # TODO: Your code here                                                    #
    ###########################################################################
    #Normalize data
    T1 = Normalize(pts1)
    T2 = Normalize(pts2)
    #get homogenous coordinate
    #pts1_ho = np.hstack([pts1, np.ones((pts1.shape[0], 1))])  
    pts1_ho = homogenize(pts1)
    pts2_ho = homogenize(pts2)
    #multiply projection matrix with pts3d  
    pts1 = T1 @  pts1_ho.T
    pts2 = T2 @ pts2_ho.T
    #normalized points
    u = pts1.T
    v = pts2.T
    
    n = u.shape[0]
    if v.shape[0] != n:
        raise ValueError("Number of points don't match.")
    A = np.zeros((len(u),9))
    # img1 and img2
    for i in range(len(u)):
        A[:,0] = u[:,0]*v[:,0]
        A[:,1] = u[:,0]*v[:,1]
        A[:,2] = u[:,0]
        A[:,3] = u[:,1]*v[:,0]
        A[:,4] = u[:,1]*v[:,1]
        A[:,5] = u[:,1] 
        A[:,6] = u[:,0]
        A[:,7] = u[:,1]
        A[:,8] = 1.0 
        
    # compute linear least square solution
    U,Su,V = np.linalg.svd(A)
    F_hat = V[-1].reshape(3,3)
    # Enforce rank contraint = 2 
    u,su,v = np.linalg.svd(F_hat)
    #dot product
    f_hat = u @ np.diag([*su[:2], 0]) @ v
    #transformation
    F = f_hat/f_hat[2,2]
    #restore 
    Fmat = T2.transpose() @ (np.dot(F, T1))
    ###########################################################################
    #                             END OF YOUR CODE                            #
    ###########################################################################
    return T1,T2, F, Fmat


def compute_epipoles(F):
    """
    Given a Fundamental Matrix F, return the epipoles represented in
    homogeneous coordinates.

    Check: e2@F and F@e1 should be close to [0,0,0]

    Inputs:
    - F: the fundamental matrix

    Return:
    - e1: the epipole for image 1 in homogeneous coordinates
    - e2: the epipole for image 2 in homogeneous coordinates
    """
    ###########################################################################
    # TODO: Your code here                                                    #
    ###########################################################################
    F = F.transpose() @ F
    w3, v3 = np.linalg.eig(F)
    small = np.argmin(w3)
    F_vect = v3[:,small]
    e1 = F_vect/F_vect[2]

    FT_mat = F @ F.transpose()
    w4 , v4 = np.linalg.eig(FT_mat)
    small = np.argmin(w4)
    F_vect = v4[:,small]
    e2 = F_vect/F_vect[2]
    
    ###########################################################################
    #                             END OF YOUR CODE                            #
    ###########################################################################

    return e1,e2
 

In [17]:
# You can run it on one or all the examples
names = os.listdir("/content/gdrive/MyDrive/3rd_semester/Computer Vision/assignments/assignment_5/data/task23")
output = "results/"

if not os.path.exists(output):
    os.mkdir(output)
Fund_mat = []
for name in names:
    # load the information
    img1 = cv2.imread(os.path.join("/content/gdrive/MyDrive/3rd_semester/Computer Vision/assignments/assignment_5/data/task23", name, "im1.png"))
    img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    img2 = cv2.imread(os.path.join("/content/gdrive/MyDrive/3rd_semester/Computer Vision/assignments/assignment_5/data/task23", name, "im2.png"))
    img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    data = np.load(os.path.join("/content/gdrive/MyDrive/3rd_semester/Computer Vision/assignments/assignment_5/data/task23", name, "data.npz"))
    pts1 = data['pts1'].astype(float)
    pts2 = data['pts2'].astype(float)
    K1 = data['K1']
    K2 = data['K2']
    shape = img1.shape
    Fund_mat.insert(0, name)
    #finding Fundamental matrix
    T1, T2, F, Fmat = find_fundamental_matrix(shape, pts1, pts2)
    Fund_mat.insert(1, F)
    Fund_mat.insert(2, Fmat)
    e1 , e2 = compute_epipoles(F)
    draw_epipolar(img1, img2, F, pts1, pts2, e1, e2, filename='xtrans')
    # you can check against this
    #FCheck, _ = cv2.findFundamentalMat(pts1, pts2, cv2.FM_8POINT)
    

    #######################################################################
    # TODO: Your code here                                                #
    #######################################################################


/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:80: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:81: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
